# Part A — Image Filtering and Enhancement

Compare Mean, Gaussian and Median denoising followed by Laplacian sharpening on the selected photograph of potted succulents. The pot surface, leaf edges and gravel provide smooth regions, strong boundaries and fine texture.

[Python implementation](../src/part_a.py) · [Dataset metadata](../data/part_a/metadata.json)


In [ ]:
import ast
import base64
import copy
import csv
import hashlib
import io
import json
import sys
from html import escape
from pathlib import Path

from IPython.display import HTML, Image as DisplayImage, display
from PIL import Image as PILImage

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / "src/part_a.py").is_file())
sys.path.insert(0, str(project_root / "src"))
import part_a

image_path = project_root / part_a.DEFAULT_INPUT
config = copy.deepcopy(part_a.DEFAULT_CONFIG)
reference, input_metadata = part_a.load_reference(image_path, config["max_long_edge"])
roi_boxes = part_a.resolve_rois(config["roi_boxes"], reference.shape)

def display_table(records, columns):
    header = "".join("<th scope='col'>" + escape(label) + "</th>" for _, label in columns)
    body = []
    for record in records:
        cells = ["<td>" + escape(str(record.get(key, "N/A"))) + "</td>" for key, _ in columns]
        body.append("<tr>" + "".join(cells) + "</tr>")
    display(HTML("<div style='overflow-x:auto'><table><thead><tr>" + header
                 + "</tr></thead><tbody>" + "".join(body) + "</tbody></table></div>"))

def show_figure(name):
    path = run_dir / "figures" / (name + ".png")
    assert path.is_file(), name
    display(DisplayImage(filename=str(path), alt=name.replace("_", " ")))

def find_case(noise_type, level, stage, method, kernel=None, sigma=None,
              laplacian_kernel=None, alpha=None):
    wanted = {"noise_type": noise_type, "noise_level": str(float(level)), "seed": "42",
              "stage": stage, "method": method,
              "denoise_kernel_size": "" if kernel is None else str(kernel),
              "filter_sigma": "" if sigma is None else str(float(sigma)),
              "laplacian_kernel_size": "" if laplacian_kernel is None else str(laplacian_kernel),
              "alpha": "" if alpha is None else str(float(alpha))}
    matches = [row for row in metric_rows if all(row[key] == value for key, value in wanted.items())]
    assert len(matches) == 1, wanted
    return matches[0]

def show_crops(cases, labels, roi_name):
    assert len(cases) == len(labels)
    box = roi_boxes[roi_name]
    paths = [run_dir / "reference.png"] + [run_dir / row["image_path"] for row in cases]
    panels = []
    for path, label in zip(paths, ["Reference"] + labels, strict=True):
        with PILImage.open(path) as image:
            crop = image.convert("RGB").crop(box)
        buffer = io.BytesIO()
        crop.save(buffer, format="PNG")
        encoded = base64.b64encode(buffer.getvalue()).decode("ascii")
        panels.append("<figure style='margin:0;flex:0 0 180px'>"
                      + "<img alt='" + escape(label) + "' src='data:image/png;base64," + encoded
                      + "' style='width:180px;image-rendering:pixelated'>"
                      + "<figcaption style='font-size:12px;text-align:center'>"
                      + escape(label) + "</figcaption></figure>")
    display(HTML("<div style='display:flex;gap:12px;overflow-x:auto;padding:8px 0'>"
                 + "".join(panels) + "</div>"))

## 1. Input and parameters

The user-provided JPEG is **4128 × 3096** with EXIF orientation 6. EXIF correction and RGB conversion precede area resizing to **768 × 1024**. The JPEG bytes are preserved; no crop is applied to the reference. Native camera grain and compression may remain in the pre-injection reference.

Three ROIs, selected before filtering, separate the **pot surface**, **leaf edges** and **gravel**.

| Parameter | Values |
|---|---|
| Gaussian noise standard deviation, $\sigma_n$ | 10, 20, 30 intensity levels |
| Salt-and-Pepper amount | 1%, 3%, 5%; equal salt/pepper proportions |
| Denoising kernels | 3×3, 5×5, 7×7, 9×9, 11×11 |
| Gaussian filter standard deviation, $\sigma_f$ | 0.6, 0.8, 1.0, 1.2, 1.4, 1.5, 1.6 |
| Laplacian kernels | 3×3, 5×5, 7×7 |
| Sharpening strength, $\alpha$ | 0.1, 0.2, 0.25, 0.5 |
| Random seeds | 42, 43, 44 |

Denoise–sharpen chains use the image-specific selected denoiser within each method at each noise level. Those choices are frozen in the run configuration. Direct-sharpening controls use $\alpha=0.25$ and $0.5$.

The notebook loads the recorded run when available. Set `rerun_experiments = True` to measure a new full run; missing recorded results also trigger a fresh run.


## 2. Methods and evaluation

The workflow is **reference → noisy image → denoising → Laplacian sharpening**. Direct controls sharpen the noisy image without denoising.

**Noise.** Gaussian noise is added independently to RGB elements in floating point, then clipped and rounded to the 0–255 range. Salt-and-Pepper noise replaces selected unique pixel positions with black or white across all three channels. Realized noise statistics are stored in [run metadata](../results/part_a/official/20261019T024705Z_aef5c7be/run.json).

**Denoising.** Mean filtering averages the neighbourhood; Gaussian filtering uses normalized weights

$$
G(x,y) \propto \exp\left(-\frac{x^2+y^2}{2\sigma_f^2}\right).
$$

Median filtering takes the neighbourhood median. All three filters use replicated borders.

**Sharpening.** The signed Laplacian is computed in float32. For a denoised image $D$,

$$
J=\operatorname{round}\left(\operatorname{clip}\left(D-\alpha s_k\Delta_kD,0,255\right)\right),
\qquad s_k=2^{4-2k},\quad k\in\{3,5,7\}.
$$

The scale normalizes derivative gain across the OpenCV apertures. Sharpening increases edge contrast and can amplify residual noise.

**Image quality.** Every output $J$ is compared with the same reference $I$:

$$
\operatorname{MSE}=\frac{1}{3HW}\sum_{y=1}^{H}\sum_{x=1}^{W}\sum_{c=1}^{3}
\left(I_{yxc}-J_{yxc}\right)^2,
\qquad
\operatorname{PSNR}=10\log_{10}\left(\frac{255^2}{\operatorname{MSE}}\right).
$$

MSE uses float64 differences; zero MSE gives infinite PSNR. RGB SSIM is averaged over channels with `data_range=255`, Gaussian window weights of sigma 1.5 and population covariance. The SSIM window sigma is separate from both noise sigma and filter sigma.

**Timing.** Each seed has two warm-ups followed by ten timed repetitions. Quality is reported as **mean ± sample standard deviation over three seeds**. Runtime is the **median (IQR) of 30 samples** for one configuration. It includes processing checks and array conversions, and excludes noise generation, metrics, file I/O and plotting. Denoising plus sharpening is timed as a complete chain. OpenCV uses one thread with OpenCL disabled; samples from different measurement runs are not pooled.

### Parameter search

The search tested **382 distinct denoiser configurations**, including **five kernel sizes** and **22 Gaussian sigma values** across coarse, fine and boundary checks. It reused earlier identical tuples and retained one timing cohort per tuple. The final comparison grid uses five kernels and seven Gaussian sigmas; the broader search remains in the linked records.

Choices maximize three-seed mean whole-image PSNR per noise level and method. Equal scores prefer smaller kernels; leaf/gravel crops expose over-smoothing.

[Combined search table](../results/part_a/succulent/parameter_search/combined_summary.csv) · [Selected parameters](../results/part_a/succulent/parameter_search/selection.json) · [Coarse-search notebook](../results/part_a/succulent/parameter_search/20261019T023103Z_af240871/experiment.ipynb) · [Refinement notebook](../results/part_a/succulent/parameter_search/20261019T023518Z_d634fae8/experiment.ipynb) · [Final-grid coverage notebook](../results/part_a/succulent/parameter_search/20261019T024108Z_3213ef5e/experiment.ipynb)


In [ ]:
selected_parameters = []
for noise_type, levels in config["noise_levels"].items():
    for level in levels:
        for method in ("mean", "gaussian", "median"):
            kernel, sigma = part_a.selected_filter(noise_type, level, method, config)
            selected_parameters.append({
                "noise": noise_type, "level": level, "method": method,
                "kernel": kernel, "sigma": sigma if sigma is not None else "—",
            })
display_table(selected_parameters, [
    ("noise", "Noise"), ("level", "Level"), ("method", "Method"),
    ("kernel", "Kernel"), ("sigma", "Filter σ"),
])

In [ ]:
cached_run = project_root / "results/part_a/official/20261019T024705Z_aef5c7be"
rerun_experiments = not cached_run.is_dir()
if rerun_experiments:
    run_dir = part_a.run_experiments(image_path, project_root / part_a.DEFAULT_OUTPUT, config)
else:
    run_dir = cached_run

manifest = json.loads((run_dir / "run.json").read_text())
assert manifest["status"] == "complete" and manifest["config"] == config
assert manifest["input"]["sha256"] == input_metadata["sha256"]
source_path = project_root / "src/part_a.py"
recorded_source = manifest["source_code"]
assert manifest["code_sha256"] == hashlib.sha256(recorded_source.encode()).hexdigest()
if rerun_experiments:
    assert manifest["code_sha256"] == hashlib.sha256(source_path.read_bytes()).hexdigest()


def measurement_syntax(source):
    # Hình vẽ và metadata môi trường không làm thay đổi số liệu đã đo.
    tree = ast.parse(source)
    tree.body = [node for node in tree.body
                 if not (isinstance(node, ast.FunctionDef)
                         and node.name in ("build_figures", "check_environment"))]
    return ast.dump(tree, include_attributes=False)


if measurement_syntax(recorded_source) != measurement_syntax(source_path.read_text()):
    raise RuntimeError("Mã xử lý hoặc tính metric đã thay đổi so với run đã lưu. "
                       "Đặt rerun_experiments = True để đo lại trước khi dùng kết quả.")

# Dựng lại hình từ số liệu đã lưu để áp dụng độ phân giải của mã hiện tại.
part_a.build_figures(run_dir, roi_boxes)
figure_receipt = json.loads((run_dir / "figures/figures.json").read_text())
assert figure_receipt["figure_code_sha256"] == hashlib.sha256(source_path.read_bytes()).hexdigest()
validation = json.loads((run_dir / "validation.json").read_text())
assert validation["valid"], validation["errors"]
metric_rows = part_a.read_csv(run_dir / "metrics.csv")
summary_rows = part_a.read_csv(run_dir / "summary.csv")
expected = part_a.expected_counts(config)
assert len(metric_rows) == expected["metrics"]
assert len(summary_rows) == expected["metrics"] // len(config["seeds"])
assert all(row["status"] == "ok" for row in metric_rows)
assert len(part_a.read_csv(run_dir / "timings.csv")) == expected["timings"]
measured_environment = manifest["environment"]
environment_rows = [{"item": "Python", "value": measured_environment["python"].split()[0]}]
environment_rows += [{"item": name, "value": version}
                     for name, version in measured_environment["packages"].items()]
display_table(environment_rows, [("item", "Environment"), ("value", "Version")])
show_figure("reference_rois")
relative_run = run_dir.relative_to(project_root).as_posix()
display(HTML("<p>Results: <a href='../" + relative_run + "/summary.csv'>"
             + str(len(summary_rows)) + " configurations</a>, " + str(len(metric_rows))
             + " seed-level metric rows and " + str(expected["timings"]) + " runtime samples.</p>"))


## 3. Denoising results

The tables compare the noisy baselines and selected parameters for all three denoisers. Quality values are mean ± standard deviation; runtime is median (IQR), in milliseconds. A noisy baseline has no filter runtime.

The full kernel/sigma sweep is in [summary.csv](../results/part_a/official/20261019T024705Z_aef5c7be/summary.csv), with [seed-level metrics](../results/part_a/official/20261019T024705Z_aef5c7be/metrics.csv) and [raw runtime samples](../results/part_a/official/20261019T024705Z_aef5c7be/timings.csv).

In [ ]:
def quality_rows(records):
    formatted = []
    for row in records:
        is_gaussian = row["noise_type"] == "gaussian"
        level = float(row["noise_level"])
        filter_name = "Noisy" if row["stage"] == "noisy" else row["method"].title()
        if row["denoise_kernel_size"]:
            filter_name += " " + row["denoise_kernel_size"] + "×" + row["denoise_kernel_size"]
        if row["filter_sigma"]:
            filter_name += ", σ=" + row["filter_sigma"]
        formatted.append({
            "noise": "Gaussian" if is_gaussian else "Salt-and-Pepper",
            "level": "σ=" + str(int(level)) if is_gaussian else f"{level:.0%}",
            "filter": filter_name,
            "psnr": f"{float(row['psnr_db_mean']):.4f} ± {float(row['psnr_db_std']):.4f}",
            "ssim": f"{float(row['ssim_mean']):.4f} ± {float(row['ssim_std']):.4f}",
            "mse": f"{float(row['mse_mean']):.2f} ± {float(row['mse_std']):.2f}",
            "runtime": (f"{float(row['runtime_ms']):.3f} ({float(row['runtime_iqr_ms']):.3f})"
                        if row["runtime_ms"] else "N/A"),
        })
    return formatted

quality_columns = [("noise", "Noise"), ("level", "Level"), ("filter", "Filter"),
                   ("psnr", "PSNR (dB)"), ("ssim", "SSIM"), ("mse", "MSE (pixel²)"),
                   ("runtime", "Runtime (ms)")]
display_table(quality_rows(part_a.read_csv(run_dir / "tables/noisy_baselines.csv")), quality_columns)
selected_rows = part_a.read_csv(run_dir / "tables/denoised_comparison.csv")
display_table(quality_rows(selected_rows), quality_columns)

### Gaussian noise

The selected Gaussian settings are **5×5, σ=0.8 / 7×7, σ=1.2 / 9×9, σ=1.5** at noise standard deviations 10/20/30. They are chosen by three-seed mean PSNR from the expanded search, with leaf/gravel metrics and crops used to inspect detail loss.

The grid uses noise σ=20 and seed 42. Whole-image tables average all three seeds. Increasing the Mean kernel illustrates how the smooth pot can become cleaner while leaf tips and gravel boundaries become less distinct.

The selected Gaussian settings yield PSNR **34.2954 / 31.2319 / 29.4227 dB** and SSIM **0.8779 / 0.8251 / 0.7942** at noise standard deviations 10/20/30.


In [ ]:
show_figure("gaussian_grid")
show_figure("gaussian_sensitivity")
show_figure("gaussian_gaussian_heatmap")
mean_kernels = [find_case("gaussian", 20, "denoised", "mean", kernel=kernel)
                for kernel in config["denoise_kernels"]]
display(HTML("<p>Mean filtering: loss of gravel texture as the kernel grows.</p>"))
show_crops(mean_kernels, [f"Mean {kernel}×{kernel}" for kernel in config["denoise_kernels"]], "gravel")

### Salt-and-Pepper noise

Median 3×3 is selected at all three impulse levels. The expanded kernel comparison uses 3/5/7/9/11 on the same gravel crop. A fixed Gaussian 5×5, σ=0.6 control at 1% impulses helps expose residual spots on the pot surface.

Median 3×3 yields PSNR **38.2479 / 37.8470 / 37.4485 dB** and SSIM **0.9651 / 0.9639 / 0.9626** at 1%/3%/5% impulses. At 3%, the noisy baseline is **20.2892 dB / 0.4403 SSIM**.


In [ ]:
show_figure("salt_pepper_grid")
show_figure("salt_pepper_sensitivity")
median_kernels = [find_case("salt_pepper", 0.03, "denoised", "median", kernel=kernel)
                  for kernel in config["denoise_kernels"]]
display(HTML("<p>Median filtering: increasing kernel size smooths gravel texture.</p>"))
show_crops(median_kernels, [f"Median {kernel}×{kernel}" for kernel in config["denoise_kernels"]], "gravel")
low_impulse = [find_case("salt_pepper", 0.01, "denoised", "gaussian", kernel=5, sigma=0.6),
               find_case("salt_pepper", 0.01, "denoised", "median", kernel=3)]
display(HTML("<p>1% impulses: residual spots in the pot surface.</p>"))
show_crops(low_impulse, ["Gaussian 5×5, σ=0.6", "Median 3×3"], "pot_wall")

## 4. Laplacian sharpening

Laplacian 3×3, α=0.1 is shown after each selected denoiser. The table reports its measured before/after fidelity. All 216 chains compare Laplacian kernels 3/5/7 and α=0.1/0.2/0.25/0.5 after the image-specific denoiser for each method and noise level.

At Gaussian noise σ=20, the mild Laplacian 3×3, α=0.1 changes PSNR **31.2319 → 31.2305 dB** and SSIM **0.8251 → 0.8144**. At 3% impulses after Median 3×3, the changes are **37.8470 → 37.9997 dB** and **0.9639 → 0.9651 SSIM**.


In [ ]:
chains = part_a.read_csv(run_dir / "tables/sharpening_comparison.csv")
preferred_chains = [row for row in chains
                    if row["method"] == ("gaussian" if row["noise_type"] == "gaussian" else "median")
                    and row["laplacian_kernel_size"] == "3" and row["alpha"] == "0.1"]
assert len(preferred_chains) == 6
sharpening_rows = []
for row in preferred_chains:
    level = float(row["noise_level"])
    sharpening_rows.append({
        "noise": "Gaussian σ=" + str(int(level)) if row["noise_type"] == "gaussian" else f"Salt-and-Pepper {level:.0%}",
        "psnr_before": f"{float(row['denoised_psnr_db_mean']):.4f}",
        "psnr_after": f"{float(row['psnr_db_mean']):.4f}",
        "ssim_before": f"{float(row['denoised_ssim_mean']):.4f}",
        "ssim_after": f"{float(row['ssim_mean']):.4f}",
        "mse": f"{float(row['mse_mean']):.2f}",
        "runtime": f"{float(row['runtime_ms']):.3f}",
    })
display_table(sharpening_rows, [("noise", "Noise"), ("psnr_before", "PSNR before"),
              ("psnr_after", "PSNR after (dB)"), ("ssim_before", "SSIM before"),
              ("ssim_after", "SSIM after"), ("mse", "MSE after"), ("runtime", "Chain time (ms)")])
show_figure("gaussian_sharpening_sensitivity")
show_figure("salt_pepper_sharpening_sensitivity")

### Edge contrast, halos and residual noise

The crops hold each selected denoiser and Laplacian kernel 3 fixed while increasing α. Compare leaf boundaries for light/dark overshoot and the gravel for harsh texture. Repeated oscillatory ringing should only be claimed if clearly visible.


In [ ]:
for noise_type, level, method, kernel, sigma in [
        ("gaussian", 20, "gaussian", *part_a.selected_filter("gaussian", 20, "gaussian", config)),
        ("salt_pepper", 0.03, "median", 3, None)]:
    denoised = find_case(noise_type, level, "denoised", method, kernel, sigma)
    alpha_cases = [find_case(noise_type, level, "denoised_then_sharpened", method,
                            kernel, sigma, laplacian_kernel=3, alpha=alpha)
                   for alpha in (0.1, 0.2, 0.25, 0.5)]
    display(HTML("<p>" + escape(noise_type.replace("_", " ").title())
                 + ": leaf edges and gravel.</p>"))
    for roi_name in ("leaf_edges", "gravel"):
        show_crops([denoised] + alpha_cases,
                   ["Denoised", "α=0.1", "α=0.2", "α=0.25", "α=0.5"], roi_name)

### Direct-sharpening control

Direct Laplacian 3×3 sharpening with α=0.5 is applied to Gaussian noise σ=20 without denoising. The pot crop separates noise amplification from the denoise–sharpen chain. All measured direct controls remain in the metrics.

Direct sharpening lowers PSNR **22.3294 → 16.4111 dB** and SSIM **0.2853 → 0.1311**.


In [ ]:
noisy = find_case("gaussian", 20, "noisy", "none")
direct = find_case("gaussian", 20, "noisy_then_sharpened", "laplacian",
                   laplacian_kernel=3, alpha=0.5)
show_crops([noisy, direct], ["Gaussian noise σ=20", "Direct Laplacian, α=0.5"], "pot_wall")

## 5. Whole-image score versus local detail

At Gaussian noise σ=20, compare the chosen PSNR-maximizing denoiser with the SSIM-maximizing denoiser across all three methods. The pot, leaf and gravel table uses **seed 42 only**, not three-seed means. Identical crop coordinates make local detail loss visible alongside whole-image scores.


In [ ]:
denoising_candidates = [row for row in summary_rows if row["noise_type"] == "gaussian"
                        and row["noise_level"] == "20.0" and row["stage"] == "denoised"]
roi_cases = []
for metric in ("psnr_db_mean", "ssim_mean"):
    best = max(denoising_candidates, key=lambda row: float(row[metric]))
    roi_cases.append(find_case("gaussian", 20, "denoised", best["method"],
                              int(best["denoise_kernel_size"]),
                              float(best["filter_sigma"]) if best["filter_sigma"] else None))
roi_records = part_a.read_csv(run_dir / "tables/roi_metrics.csv")
roi_index = {(row["case_id"], row["roi"]): row for row in roi_records}
assert len(roi_records) == len(roi_index) == sum(row["image_saved"] == "true" for row in metric_rows) * len(roi_boxes)
roi_comparison = []
roles = ["Highest whole-image PSNR", "Highest whole-image SSIM"]
for role, case in zip(roles, roi_cases, strict=True):
    for roi_name in roi_boxes:
        measured = roi_index[(case["case_id"], roi_name)]
        assert measured["status"] == "ok"
        roi_comparison.append(dict(measured, role=role, method=case["method"],
            kernel=case["denoise_kernel_size"], sigma=case["filter_sigma"]))
roi_comparison_path = run_dir / "tables/objective_roi_comparison.csv"
with roi_comparison_path.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(roi_comparison[0]))
    writer.writeheader()
    writer.writerows(roi_comparison)
assert len(roi_comparison) == 6
shown_roi = [dict(row, psnr_db=f"{float(row['psnr_db']):.4f}", ssim=f"{float(row['ssim']):.4f}")
             for row in roi_comparison]
display_table(shown_roi, [("role", "Objective"), ("method", "Filter"), ("kernel", "Kernel"),
              ("sigma", "Filter σ"), ("roi", "ROI"), ("psnr_db", "PSNR (dB)"), ("ssim", "SSIM")])
labels = []
for role, case in zip(roles, roi_cases, strict=True):
    label = role + ": " + case["method"].title() + " " + case["denoise_kernel_size"] + "×" + case["denoise_kernel_size"]
    if case["filter_sigma"]:
        label += ", σ=" + case["filter_sigma"]
    labels.append(label)
for roi_name in roi_boxes:
    display(HTML("<p>" + escape(roi_name.replace("_", " ").title()) + "</p>"))
    show_crops(roi_cases, labels, roi_name)


## 6. Selected outputs within the measured grid

The following recipes maximize three-seed mean PSNR among the measured denoised outputs and denoise–sharpen chains at each noise level. Direct-sharpening controls are excluded. Laplacian parameters are blank when the denoised image itself is selected. 

Quality values average three seeds; the linked PNG examples use seed 42. Crop inspection remains necessary when fidelity and edge contrast favour different outputs.


In [ ]:
recommended = []
recommended_view = []
for noise_type, levels in config["noise_levels"].items():
    for level in levels:
        candidates = [row for row in summary_rows if row["noise_type"] == noise_type
                      and float(row["noise_level"]) == level
                      and row["stage"] in ("denoised", "denoised_then_sharpened")]
        best = max(candidates, key=lambda row: float(row["psnr_db_mean"]))
        examples = [row for row in metric_rows if row["seed"] == "42"
                    and all(row[field] == best[field] for field in part_a.GROUP_FIELDS)]
        assert len(examples) == 1
        example = examples[0]
        assert example["image_saved"] == "true"
        image_file = run_dir / example["image_path"]
        assert hashlib.sha256(image_file.read_bytes()).hexdigest() == example["image_sha256"]
        record = dict(best, example_seed=42, image_path=example["image_path"],
                      image_sha256=example["image_sha256"], case_id=example["case_id"])
        recommended.append(record)
        shown = quality_rows([best])[0]
        shown.update(laplacian=best["laplacian_kernel_size"] or "—", alpha=best["alpha"] or "—")
        recommended_view.append(shown)
assert len(recommended) == 6
recommended_path = run_dir / "tables/recommended_outputs.csv"
with recommended_path.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(recommended[0]))
    writer.writeheader()
    writer.writerows(recommended)
display_table(recommended_view, [("noise", "Noise"), ("level", "Level"), ("filter", "Denoiser"),
              ("laplacian", "Laplacian kernel"), ("alpha", "α"), ("psnr", "PSNR (dB)"),
              ("ssim", "SSIM"), ("mse", "MSE"), ("runtime", "Runtime (ms)")])
for row in recommended:
    level = float(row["noise_level"])
    label = "Gaussian σ=" + str(int(level)) if row["noise_type"] == "gaussian" else f"Salt-and-Pepper {level:.0%}"
    display(HTML("<h4>" + escape(label) + "</h4>"))
    display(DisplayImage(filename=str(run_dir / row["image_path"]), alt=label))


## 7. Parameter choice

The expanded search selects **Gaussian 5×5, σ=0.8** at Gaussian noise σ=10, **7×7, σ=1.2** at σ=20 and **9×9, σ=1.5** at σ=30. **Median 3×3** is selected for all three impulse levels. These choices prioritize mean PSNR while the ROI comparison checks edge/texture loss.

Mean/Median controls also use image-specific parameters for sharpening rather than always using kernel 3.


## References

- [OpenCV image filtering](https://docs.opencv.org/4.x/d4/d86/group__imgproc__filter.html)
- [OpenCV Laplacian operator](https://docs.opencv.org/4.x/d5/db5/tutorial_laplace_operator.html)
- [scikit-image quality metrics](https://scikit-image.org/docs/stable/api/skimage.metrics.html#skimage.metrics.structural_similarity)
